In [13]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [14]:
# Data and model path
ML_DIR = Path.cwd().parent
DATA_DIR = ML_DIR / 'data' / 'iris'
MODELS_DIR = ML_DIR.parent / 'include' / 'models' / 'iris'

In [15]:
# Model parameters
TEST_SIZE = 0.2
EPOCHS = 300
LEARNING_RATE = 0.01

In [16]:
# Load the dataset
df = pd.read_csv(DATA_DIR / 'iris.csv')

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (150, 6)
   id  sepal_length  sepal_width  petal_length  petal_width species
0   1           5.1          3.5           1.4          0.2  setosa
1   2           4.9          3.0           1.4          0.2  setosa
2   3           4.7          3.2           1.3          0.2  setosa
3   4           4.6          3.1           1.5          0.2  setosa
4   5           5.0          3.6           1.4          0.2  setosa


In [17]:
# Extract features
feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_columns].to_numpy()

# Encode labels
label_mapping = {"setosa": 0, "versicolor": 1, "virginica": 2}
y = df["species"].map(label_mapping).values

if pd.isna(y).any():
    raise ValueError("Unknown class found in Species column.")

print(f"Class distribution: {df["species"].value_counts()}")

Class distribution: species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [18]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y,
)

In [19]:
# Scale the features
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [20]:
# Define the NN model
class IrisNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 8),    # Input layer (4 features) to hidden layer (8 neurons)
            nn.ReLU(),          # RELU activation function
            nn.Linear(8, 3),    # Hidden layer (8 neurons) to output layer (3 classes)
            nn.Softmax(dim=1)   # Softmax activation for multi-class classification
        )

    def forward(self, x):
        return self.network(x)


model = IrisNet()

print(f"Model: {model}")

Model: IrisNet(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=3, bias=True)
    (3): Softmax(dim=1)
  )
)


In [21]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [22]:
# Training loop
print("\nTraining...")
for epoch in range(EPOCHS):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1:3d}/{EPOCHS}] "
            f"Loss: {loss.item():.4f}"
        )


Training...
Epoch [ 10/300] Loss: 1.0907
Epoch [ 20/300] Loss: 1.0339
Epoch [ 30/300] Loss: 0.9326
Epoch [ 40/300] Loss: 0.8423
Epoch [ 50/300] Loss: 0.7802
Epoch [ 60/300] Loss: 0.7296
Epoch [ 70/300] Loss: 0.6868
Epoch [ 80/300] Loss: 0.6548
Epoch [ 90/300] Loss: 0.6329
Epoch [100/300] Loss: 0.6183
Epoch [110/300] Loss: 0.6085
Epoch [120/300] Loss: 0.6018
Epoch [130/300] Loss: 0.5971
Epoch [140/300] Loss: 0.5938
Epoch [150/300] Loss: 0.5912
Epoch [160/300] Loss: 0.5891
Epoch [170/300] Loss: 0.5875
Epoch [180/300] Loss: 0.5861
Epoch [190/300] Loss: 0.5849
Epoch [200/300] Loss: 0.5839
Epoch [210/300] Loss: 0.5830
Epoch [220/300] Loss: 0.5823
Epoch [230/300] Loss: 0.5816
Epoch [240/300] Loss: 0.5809
Epoch [250/300] Loss: 0.5804
Epoch [260/300] Loss: 0.5799
Epoch [270/300] Loss: 0.5794
Epoch [280/300] Loss: 0.5790
Epoch [290/300] Loss: 0.5786
Epoch [300/300] Loss: 0.5782


In [23]:
# Evaluate the model
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    predictions = torch.argmax(outputs, dim=1)

accuracy = accuracy_score(y_test.numpy(), predictions.numpy())

print(f"Test accuracy: {accuracy * 100:.4f}%")

Test accuracy: 100.0000%


In [24]:
# Convert the model to AIfES model and save it
from aifes import pytorch2aifes

pytorch2aifes.convert_to_fnn_f32_express(model, str(MODELS_DIR))